In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [30]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [31]:
import os
BASE_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
print("Searching existing model files...\n")
for root, dirs, files in os.walk(BASE_PATH):
    if "day22_outputs" in root:
        continue
    for file in files:
        if file.lower().endswith(".joblib"):
            print(os.path.join(root, file))

Searching existing model files...

/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/SOC_preprocessing_pipeline_day11.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/day11_preprocessing_metadata.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day11_outputs/remaining_time_preprocessing_pipeline_day11.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day12_dataset_V1/SOC_preprocessing_pipeline_day12_V1.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day12_dataset_V1/remaining_time_preprocessing_pipeline_day12_V1.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day15_outputs/SOC_linear_regression_baseline_day15.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day15_outputs/Remaining_Time_linear_regression_baseline_day15.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day16_outputs/RandomForest_SOC_day16.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day16_outputs/RandomForest_Remaining_Time_day16.joblib
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day16_out

In [32]:
import os
import shutil
import json
from datetime import datetime
BASE_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
DAY19_PATH = os.path.join(BASE_PATH, "day19_outputs")
DAY20_PATH = os.path.join(BASE_PATH, "day20_outputs")
DAY22_PATH = os.path.join(BASE_PATH, "day22_outputs")
os.makedirs(DAY22_PATH, exist_ok=True)
print("DAY 22 — FINAL MODEL / PIPELINE FREEZE")

DAY 22 — FINAL MODEL / PIPELINE FREEZE


In [33]:
SOC_MODEL = os.path.join(
    DAY19_PATH,"day19_best_SOC_XGBoost.joblib")
REMAINING_MODEL = os.path.join(DAY19_PATH,"day19_best_Remaining_Charging_Time_Random_Forest.joblib")
print("\nChecking final model files...")
if not os.path.exists(SOC_MODEL):
    raise FileNotFoundError(f"SOC model not found:\n{SOC_MODEL}")
if not os.path.exists(REMAINING_MODEL):
    raise FileNotFoundError(
        f"Remaining Charging Time model not found:\n{REMAINING_MODEL}")
print("SOC XGBoost model found")
print("Remaining Charging Time Random Forest model found")


Checking final model files...
SOC XGBoost model found
Remaining Charging Time Random Forest model found


In [34]:
soc_final = os.path.join(DAY22_PATH,"FINAL_SOC_XGBoost_day22.joblib")
remaining_final = os.path.join(DAY22_PATH,"FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib")
shutil.copy2(SOC_MODEL, soc_final)
shutil.copy2(REMAINING_MODEL, remaining_final)
print("\nFinal SOC model copied")
print("Final Remaining Charging Time model copied")


Final SOC model copied
Final Remaining Charging Time model copied


In [35]:
final_features = [
    "Voltage","Current","Temperature","Elapsed Time","Cycle","WhAccu","Power","Capacity"]
feature_file = os.path.join(
    DAY22_PATH,"day22_final_features.json")
with open(feature_file, "w") as f:
    json.dump(final_features, f, indent=4)
print("\nFinal feature list saved")


Final feature list saved


In [36]:
model_selection = {
    "SOC_task": {
        "final_model": "XGBoost","artifact": "FINAL_SOC_XGBoost_day22.joblib","selection_source": "Day 19 model selection / Day 20 validation evidence","test_R2": 0.990521},
    "Remaining_Charging_Time_task": {"final_model": "Random Forest","artifact": "FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib","selection_source": "Day 19 model selection / Day 20 validation evidence","test_R2": 0.998756,"validation_R2": 0.996502}
}
selection_file = os.path.join(DAY22_PATH,"day22_model_selection.json")
with open(selection_file, "w") as f:
    json.dump(model_selection, f, indent=4)
print("Model selection record saved")

Model selection record saved


In [37]:
evidence_path = os.path.join(DAY22_PATH,"previous_validation_evidence")
if os.path.exists(DAY20_PATH):
    if os.path.exists(evidence_path):
        shutil.rmtree(evidence_path)
    shutil.copytree(DAY20_PATH, evidence_path)
    print("Day 20 validation evidence preserved")
else:
    print("Day 20 folder not found — no copy made")

Day 20 validation evidence preserved


In [44]:
freeze_record = {
    "day": 22,
    "purpose": "Final Model and Pipeline Freeze",
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "final_models": {
        "SOC": "FINAL_SOC_XGBoost_day22.joblib","Remaining Charging Time":"FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib"},
    "features": final_features,
    "methodology": {
        "retraining": False,
        "new_predictions": False,
        "new_error_analysis": False,
        "model_comparison_repeated": False,
        "previous_validation_evidence": "Day 20"
    },

    "status": "FROZEN"
}
freeze_record_file = os.path.join(
    DAY22_PATH,
    "DAY22_FREEZE_RECORD.json"
)
with open(freeze_record_file, "w") as f:
    json.dump(freeze_record, f, indent=4)
print("Freeze record saved")

Freeze record saved


In [45]:
submission_text = """
DAY 22 — FINAL MODEL / PIPELINE FREEZE
Final selected models:
1. SOC Prediction
   Model: XGBoost
   Artifact: FINAL_SOC_XGBoost_day22.joblib
   Test R²: 0.990521
2. Remaining Charging Time Prediction
   Model: Random Forest
   Artifact: FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib
   Test R²: 0.998756
   Validation R²: 0.996502
Final feature set:
Voltage
Current
Temperature
Elapsed Time
Cycle
WhAccu
Power
Capacity
Day 20 validation and error-analysis outputs were preserved
as supporting evidence.
No model retraining, new prediction generation, or repeated
error analysis was performed in Day 22.
The final model artifacts and feature configuration are now frozen
for subsequent validation, technical review, packaging, and
presentation stages.
"""
submission_file = os.path.join(
    DAY22_PATH,"DAY22_SUBMISSION_NOTE.txt")
with open(submission_file, "w") as f:
    f.write(submission_text.strip())
print("Submission note saved")

Submission note saved


In [47]:
print("\nFiles currently in day22_outputs:")
for file in sorted(os.listdir(DAY22_PATH)):
    print(" •", file)
print("\nFINAL STATUS: MODEL / PIPELINE FROZEN")
print("No retraining was performed.")
print("No new predictions were generated.")
print("Day 20 validation evidence was preserved.")
print("Freeze record saved as DAY22_FREEZE_RECORD.json")


Files currently in day22_outputs:
 • DAY22_FREEZE_RECORD.json
 • DAY22_SUBMISSION_NOTE.txt
 • FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib
 • FINAL_SOC_XGBoost_day22.joblib
 • WEEK_4_DAY_23_SREEJA SANYAL.ipynb
 • day22_final_features.json
 • day22_model_selection.json
 • previous_validation_evidence

FINAL STATUS: MODEL / PIPELINE FROZEN
No retraining was performed.
No new predictions were generated.
Day 20 validation evidence was preserved.
Freeze record saved as DAY22_FREEZE_RECORD.json
